# منجّم: توقيت مصحف كامل · Munajjam: time a whole mushaf

نفس مدخلات ومخرجات نوتبوك محرك عثمان، والتوقيت بمحرك [منجّم](https://github.com/Itqan-community/Munajjam) (MIT، مجتمع إتقان).
Same inputs and output format as the Othman Engine notebook, timed with Munajjam instead.

**قبل البدء · Before you start**
1. **Runtime ← Change runtime type ← T4 GPU** (منجّم أسرع بكثير على كرت الشاشة · much faster on a GPU).
2. أضف `GH_TOKEN` من 🔑 ما دام مستودع othman-engine خاصاً (لأداة الحفظ المشتركة فقط) · only while othman-engine is private.

**إذا انقطعت الجلسة · If the session drops:** شغّل كل الخلايا من جديد (Runtime ← Run all).
السور المحفوظة في Drive تُتخطى، ويكمل من أول سورة لم تكتمل. السورة التي فشلت تُعاد عند التشغيل التالي.
Run all cells again: surahs already in Drive are skipped and it resumes from the first unfinished one.

## الإعدادات · Settings
- `SOURCES`: سطر لكل رابط. **رابط مجلد** ينتهي بـ `/` تُضاف له السور من `SURAHS` (`001.mp3` ...)،
  أو **رابط ملف** واحد ورقم السورة من اسمه (`002.mp3`).
- **مصحف عندك:** ارفع مجلده إلى Google Drive واكتب مساره، مثل `/content/drive/MyDrive/مصاحف/عبدالعزيز التركي`.
  السورة تُعرف من اسم كل ملف: رقم (`002.mp3`) أو «سورة عبس»، واسم المجلد هو اسم القارئ في النتائج.
- القيم الحالية هي عينة المقارنة (3 قراء × 5 سور). للمصحف كاملاً اترك رابط قارئ واحد واكتب `SURAHS = "1-114"`.
- الناتج: `OUT_DIR/<المحرك>/<القارئ>/002.json`، يُحفظ بعد كل سورة.

One line per URL: a **folder** ending in `/` (surahs from `SURAHS` are added) or a single **file** (surah from its name),
or a Google Drive folder path (the surah is read from each file name: a number or «سورة <name>»).
The defaults are the comparison sample. Results are saved to Drive after every surah.

`PIPELINES` · طريقتا تشغيل في منجّم، تعملان بالترتيب في نفس التشغيلة:
- `"library"`: طريقة المكتبة وسطر الأوامر في README (نموذج Whisper قرآني صغير `faster-whisper-base-ar-quran` ثم خوارزميات المحاذاة `auto`).
- `"whisperx"`: طريقة خادمهم الموصى بها في Docker (WhisperX بنموذج `large-v2` وتقطيع الآيات الخاص به). أبطأ وأثقل.

كل طريقة تُحفظ في مجلد منفصل (`munajjam` و`munajjam-whisperx`)، والمقارنة تعرض ما يوجد منهما.

In [ ]:
SOURCES = """
https://cdn.mp3quran.net/audio/abdulrahman-sudais/r1/
https://cdn.mp3quran.net/audio/mishary-alafasy/r1/
https://cdn.mp3quran.net/audio/maher-muaiqly/r3/
"""
SURAHS = "2,18,67,78,97"  # للمصحف كاملاً · whole mushaf: "1-114"
OUT_DIR = "/content/drive/MyDrive/othman-timings"
PIPELINES = ["library", "whisperx"]  # الطريقتان في تشغيلة واحدة · both pipelines in one run

## 1. التجهيز · Setup
نحو 3–5 دقائق. **أول مرة تُعاد الجلسة تلقائياً بعد التثبيت: انتظر ثواني ثم Run all مرة ثانية.**
About 3–5 minutes. The first time, the session restarts itself after installing: then Run all again.

In [ ]:
import os
import sys
from google.colab import drive, userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


drive.mount("/content/drive")
BRANCH = "main"
# GH_TOKEN مطلوب فقط ما دام المستودع خاصاً · only needed while the repository is private
gh = secret("GH_TOKEN")
repo = f"https://{gh}@github.com/vvd7mvv/othman-engine" if gh else "https://github.com/vvd7mvv/othman-engine"
!rm -rf /content/othman-engine && git clone -q --depth 1 -b {BRANCH} {repo} /content/othman-engine
sys.path.insert(0, "/content/othman-engine/notebooks")
sys.modules.pop("timings", None)  # عند إعادة التشغيل في نفس الجلسة: نحمّل النسخة الجديدة لا القديمة من الذاكرة
import timings

MUNAJJAM_COMMIT = "c8012e12cb3d335728bd1ed61de38143d1b7781a"  # مثبّت لتتكرر النتيجة · pinned for reproducibility
!rm -rf /content/munajjam && git clone -q https://github.com/Itqan-community/Munajjam /content/munajjam
!git -C /content/munajjam checkout -q {MUNAJJAM_COMMIT}
# -e لأن حزمتهم تُسقط مجلد munajjam/models (يستثنيه .gitignore عندهم)، و--no-deps لأن قيد numpy<2 يكسر بيئة Colab
# (جُرّب على numpy 2 ويعمل). av<16: الإصدارات الأحدث تكسر faster-whisper 1.2.
!pip install -q pydantic-settings librosa pydub rapidfuzz soundfile sherpa-onnx ctc-segmentation faster-whisper "av<16"
!pip install -q --no-deps -e /content/munajjam/munajjam
# whisperx يغيّر إصدار torch، فيُثبَّت دائماً قبل أول import torch (تثبيته بعدها في نفس الجلسة يكسرها)
!pip install -q git+https://github.com/m-bain/whisperx.git "av<16"
sys.path.insert(0, "/content/munajjam/munajjam")

# التثبيت قد يغيّر numpy وهي محمّلة مسبقاً في Colab، فيظهر خطأ "cannot import name '_slice'".
# الحل إعادة تشغيل الجلسة مرة واحدة؛ في المرة الثانية لا يتغير شيء فيكمل.
import numpy
installed = !python -c "import numpy; print(numpy.__version__)"
if installed[-1].strip() != numpy.__version__:
    print("🔄 تُعاد الجلسة الآن مرة واحدة بعد التثبيت. انتظر ثواني ثم Runtime ← Run all مرة ثانية")
    print("🔄 Restarting the session once after installing. Then Runtime → Run all again")
    os.kill(os.getpid(), 9)

import torch
if not torch.cuda.is_available():
    raise SystemExit("⛔ لا يوجد كرت شاشة: Runtime ← Change runtime type ← T4 GPU ← Save، ثم Run all · No GPU: switch to T4 GPU and Run all")

## 2. التوقيت · Align
أول سورة تحمّل النموذج (دقيقة أو أكثر). قبل أي سورة أطول من نصف ساعة يكتب الوقت المتوقع.
The first surah downloads the model. Before any surah longer than 30 minutes it prints the expected time.

In [ ]:
import gc

from munajjam.config import get_settings
from munajjam.core import align as munajjam_align
from munajjam.data import load_surah_ayahs
from munajjam.transcription.whisperFactory import WhisperBackend, WhisperFactory

device = torch.cuda.get_device_name(0)
settings = get_settings()
sources = timings.parse_sources(SOURCES, SURAHS)


def align_library(path, surah):
    segments = transcriber.transcribe(str(path), surah_id=surah)
    results = munajjam_align(str(path), segments, load_surah_ayahs(surah), strategy="auto")
    ayahs = {}
    for result in results:
        ayahs.setdefault(result.ayah.ayah_number, (result.start_time, result.end_time, result.similarity_score))
    return ayahs


def align_whisperx(path, surah):
    ayahs = {}
    for segment in transcriber.transcribe(str(path), surah_id=surah):
        if segment.id and 1 <= segment.id <= timings.AYAH_COUNTS[surah - 1]:
            ayahs.setdefault(segment.id, (segment.start, segment.end, getattr(segment, "confidence", None) or 0.0))
    return ayahs


for pipeline in PIPELINES:
    print(f"\n===== {pipeline} on {device} =====")
    if pipeline == "library":
        transcriber = WhisperFactory().create_whisper(
            backend=WhisperBackend.FASTERWHISPER, model_name=settings.model_id, device=settings.device)
        align, folder = align_library, "munajjam"
    else:
        transcriber = WhisperFactory().create_whisper(
            backend=WhisperBackend.WHISPERX, model_name=settings.whisperx_model_size, device=settings.device)
        align, folder = align_whisperx, "munajjam-whisperx"
    failures = timings.run_batch(sources, OUT_DIR, folder, align, device,
                                 version=f"{MUNAJJAM_COMMIT[:7]} {pipeline}", speed_guess=0.1)
    del transcriber
    gc.collect()
    torch.cuda.empty_cache()